# 🏭 Corrida forge-v1 — a forja prova que evolui política melhor que v0? (Degrau 2, issue #6)

Notebook da **corrida oficial da forja de políticas-código** (épica #1), no protocolo **pré-registrado** da issue #6 — que não muda depois do resultado:

- **A corrida:** ~300 candidatos via OpenEvolve (`run_evolution`, semente `forge/seed_policy.py`, avaliador `forge/evaluator.py`, config `forge/config.yaml`), cascata stage1 (8 episódios, campo F0) → stage2 (64 episódios, F0+F1) com limiar 0,45, ensemble LLM (Flash-Lite 0,8 + Flash 0,2) pelo segredo `GEMINI_API_KEY` do Kaggle.
- **O mutador é LLM, não programático.** O smoke (#5) provou o pipeline com mutador programático; esta corrida usa o LLM da decisão do fundador (pesquisa em `docs/PESQUISA-MUTADOR-FORGE-V1.md`). Sem o segredo, o notebook falha explícito — sem fallback silencioso.
- **Avaliação final cega, fora do loop:** o campeão, a semente v0 e o genoma campeão CMA-ES (`data/cmaes_champion.json`, `evo:6a94b587701d`) correm os MESMOS blocos cegos por `forge/blind.py` (seeds de TESTE 9901–9932, campos F0+F2 com choques, `final=True`); o veredito usa o MELHOR baseline com delta pareado de verdade contra ele. Concentração por arquétipo = delta por célula comportamental (elo × instalações).
- **Portões de produto (as 3 condições de `tournament.gates_ok`):** ecologia do campeão com eventos narráveis e diversidade comportamental ≥ 80% dos da v0, e nenhuma célula comportamental com > 50% das empresas (`forge/blind.py::product_gates`).
- **Veredito (impresso pelo notebook, `forge/verdict.py`):** PROVA (ΔS ≥ +0,08, IC95 > 0, portões ok) → Degrau 2 vira arquitetura oficial; MATA (ΔS ≤ +0,03 ou portão violado) → investimento volta ao Degrau 1; PARCIAL ( resto) → evolução por arquétipo.
- **Entregável obrigatório:** os 5 melhores programas com diff comentado contra a heurística original — saem do `evolution_trace` da corrida no fechamento da issue; este notebook salva o campeão, o veredito e o mapa dos outputs.

Regra do fundador: experimento roda AQUI, no Kaggle — nunca na VM de desenvolvimento. Pré-requisito de lançamento: bundle novo (src12) a partir da branch `forja/corrida-v1-prep` + segredo `GEMINI_API_KEY` criado pelo fundador.

In [ ]:
# 1) Fonte do pacote: dataset privado do bundle (src12, a criar no
#    lancamento a partir da branch forja/corrida-v1-prep).
#    LOADER RECURSIVO: o Kaggle expoe os datasets em
#    /kaggle/input/datasets/<dono>/<slug>/ com a arvore ja extraida;
#    a busca aceita qualquer 'src/mundo/__init__.py' sob /kaggle/input,
#    com fallback para o zip mundo-src.zip. O bundle leva src/ +
#    forge/ + pyproject.toml — a raiz do bundle entra no sys.path
#    porque a forja vive em forge/, fora do pacote.
import glob, os, sys, zipfile

inits = sorted(glob.glob('/kaggle/input/**/src/mundo/__init__.py',
                         recursive=True))
zip_cands = sorted(glob.glob('/kaggle/input/**/mundo*.zip', recursive=True))
if inits:
    bundle_root = os.path.dirname(os.path.dirname(
        os.path.dirname(inits[-1])))
elif zip_cands:
    zipfile.ZipFile(zip_cands[-1]).extractall('/kaggle/working/pkg')
    bundle_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada (loader recursivo): '
                     'anexe o dataset do bundle src12 a este kernel.')
print('bundle em', bundle_root)
sys.path.insert(0, os.path.join(bundle_root, 'src'))
sys.path.insert(0, bundle_root)
import mundo
print('mundo', mundo.__version__, 'importado de', bundle_root)
for rel in ('forge/evaluator.py', 'forge/seed_policy.py',
            'forge/config.yaml', 'forge/blind.py', 'forge/verdict.py',
            'data/cmaes_champion.json', 'src/mundo/program.py'):
    assert os.path.exists(os.path.join(bundle_root, rel)), \
        f'{rel} ausente no bundle'
print('forge/ completa no bundle')

In [ ]:
# 2) A corrida: ~300 candidatos via OpenEvolve + LLM (protocolo #6).
#    O mutador LLM exige a chave do fundador (segredo GEMINI_API_KEY
#    no Kaggle). Sem ela, falha explicita: a #6 nao tem fallback
#    programatico silencioso — o smoke (#5) ja provou o pipeline.
import json, os, subprocess, sys, time

import yaml

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openevolve'],
               check=True)
from openevolve import run_evolution

try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    raise SystemExit(
        'Sem chave LLM: crie o segredo GEMINI_API_KEY no Kaggle. '
        'A corrida #6 depende da decisao do mutador do fundador.')

with open(os.path.join(bundle_root, 'forge', 'config.yaml'),
          encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg.setdefault('llm', {})['api_key'] = api_key
run_config = '/kaggle/working/forge-v1-config.yaml'
with open(run_config, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True)

t0 = time.time()
result = run_evolution(
    initial_program=os.path.join(bundle_root, 'forge', 'seed_policy.py'),
    evaluator=os.path.join(bundle_root, 'forge', 'evaluator.py'),
    config=run_config,
    iterations=300,
    output_dir='/kaggle/working/forge-v1')
print(f'corrida concluida em {time.time() - t0:.0f}s')
print(f'melhor S no loop: {result.best_score}')
# best_program é um objeto Program no OpenEvolve 0.4.0 (não um
# caminho): o programa sai de best_code, explicitamente.
best_path = '/kaggle/working/forge-v1/best_program.py'
with open(best_path, 'w', encoding='utf-8') as f:
    f.write(result.best_code)
print('melhor programa em', best_path)

In [ ]:
# 3) AVALIACAO FINAL CEGA + VEREDITO (protocolo #6, fora do loop).
#    O campeao, a semente v0 e o genoma campeao CMA-ES correm os
#    MESMOS blocos cegos (seeds 9901-9932, F0 + F2 com choques).
#    O veredito usa o MELHOR baseline (maior S cego) com o delta
#    pareado DE VERDADE contra ele — sem ajuste improvisado.
#    Concentracao por arquétipo = delta por célula comportamental
#    (elo x instalações, eixos do MAP-Elites): concentrado se o
#    ganho vive numa única célula.
import importlib.util, json, os, shutil, statistics


def _load(name, rel):
    spec = importlib.util.spec_from_file_location(
        name, os.path.join(bundle_root, rel))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


blind = _load('forge_blind', 'forge/blind.py')
verdict_mod = _load('forge_verdict', 'forge/verdict.py')
seed_path = os.path.join(bundle_root, 'forge', 'seed_policy.py')

with open(os.path.join(bundle_root, 'data', 'cmaes_champion.json'),
          encoding='utf-8') as f:
    cmaes = json.load(f)
print(f"baseline CMA-ES: {cmaes['policy_id']} (S de selecao "
      f"{cmaes['selection_S']})")

champ_rows = blind.blind_evaluate_program(best_path, final=True, processes=1)
v0_rows = blind.blind_evaluate_program(seed_path, final=True, processes=1)
cmaes_rows = blind.blind_evaluate_genome(cmaes['genome'], final=True,
                                           processes=1)
delta_v0 = blind.paired_delta(champ_rows, v0_rows)
delta_cmaes = blind.paired_delta(champ_rows, cmaes_rows)
champ_mean = statistics.fmean(r['S'] for r in champ_rows)
v0_mean = statistics.fmean(r['S'] for r in v0_rows)
cmaes_mean = statistics.fmean(r['S'] for r in cmaes_rows)
if cmaes_mean >= v0_mean:
    eff, eff_mean = 'campeao CMA-ES', cmaes_mean
    delta_s, ci_low, eff_rows = (delta_cmaes['delta_s'],
                                 delta_cmaes['ci_low'], cmaes_rows)
else:
    eff, eff_mean = 'v0 (semente)', v0_mean
    delta_s, ci_low, eff_rows = (delta_v0['delta_s'],
                                 delta_v0['ci_low'], v0_rows)

champ_eco = blind.blind_ecology_program(best_path, processes=1)
v0_eco = blind.blind_ecology_program(seed_path, processes=1)
gates_ok, gate_reasons = blind.product_gates(champ_eco, v0_eco)

concentrated, cell_means = blind.concentrated_by_cell(champ_rows,
                                                       eff_rows)

v = verdict_mod.forge_verdict(delta_s, ci_low, gates_ok=gates_ok,
                                archetype_concentrated=concentrated)
print('=' * 64)
print(f"VEREDITO FORGE-V1: {v['verdict']}")
print(f"motivo: {v['reason']}")
print(f'campeao S cego: {champ_mean:.4f} | v0 S cego: {v0_mean:.4f} | '
      f'CMA-ES S cego: {cmaes_mean:.4f}')
print(f'baseline efetivo: {eff} (S {eff_mean:.4f}) | delta: {delta_s:+.4f} '
      f'| IC95 baixo: {ci_low:+.4f}')
print(f'delta vs v0: {delta_v0["delta_s"]:+.4f} '
      f'(IC95 baixo {delta_v0["ci_low"]:+.4f}) | delta vs CMA-ES: '
      f'{delta_cmaes["delta_s"]:+.4f} (IC95 baixo '
      f'{delta_cmaes["ci_low"]:+.4f})')
print(f"portoes (3 condicoes): {'ok' if gates_ok else gate_reasons}")
print(f'concentrado por celula: {concentrated} | '
      f'celulas com ganho: '
      f"{sum(1 for m in cell_means.values() if m > 0)} de {len(cell_means)}")
print('=' * 64)
summary = {
    'verdict': v['verdict'],
    'reason': v['reason'],
    'champ_blind_S': champ_mean,
    'v0_blind_S': v0_mean,
    'cmaes_blind_S': cmaes_mean,
    'effective_baseline': eff,
    'delta_s': delta_s,
    'ci_low': ci_low,
    'delta_vs_v0': delta_v0,
    'delta_vs_cmaes': delta_cmaes,
    'n_blocks': delta_v0['n_blocks'],
    'gates_ok': gates_ok,
    'gate_reasons': gate_reasons,
    'concentrated_by_cell': concentrated,
    'cell_means': {f'{k[0]}x{k[1]}': m for k, m in cell_means.items()},
    'champ_ecology': champ_eco,
    'v0_ecology': v0_eco,
    'best_score_loop': result.best_score,
}
with open('/kaggle/working/forge-v1-verdict.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=1)
shutil.copy(best_path, '/kaggle/working/forge-v1-best_program.py')
print('veredito salvo em /kaggle/working/forge-v1-verdict.json')
print('trace e checkpoints em /kaggle/working/forge-v1 — os 5 melhores '
      'programas com diff comentado saem do evolution_trace no '
      'fechamento da issue.')
